# Optimization · Module 08
Python 3 and NumPy only. This module comes before the classifier in Module 07, so that training is already familiar when you meet it. The objectives here are ones whose answers you can check: a quadratic and a least-squares fit.


In [1]:
import numpy as np
np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(208)
print('L(w) = (w-3)^2, starting at w = 0')
for lr in [.1, .5, .9, 1., 1.1]:
    w = 0.; path = []
    for step in range(6):
        path.append(round(w, 3)); w -= lr * 2 * (w - 3)
    print(f'learning rate {lr:3.1f}: {path}')
w = 0.; w -= .1 * 2 * (w - 3)
assert np.isclose(w, .6)
# Each step multiplies the error by (1 - 2*lr), so this quadratic converges only for 0 < lr < 1.


L(w) = (w-3)^2, starting at w = 0
learning rate 0.1: [0.0, 0.6, 1.08, 1.464, 1.771, 2.017]
learning rate 0.5: [0.0, 3.0, 3.0, 3.0, 3.0, 3.0]
learning rate 0.9: [0.0, 5.4, 1.08, 4.536, 1.771, 3.983]
learning rate 1.0: [0.0, 6.0, 0.0, 6.0, 0.0, 6.0]
learning rate 1.1: [0.0, 6.6, -1.32, 8.184, -3.221, 10.465]


## Part 2: A stretched bowl
With curvatures 1 and kappa, the steep direction limits the learning rate to below 2/kappa, while the shallow direction wants a large step. Momentum builds speed along the shallow direction. Predict both results before running the cell.


In [2]:
def run(kappa, lr, beta=0., steps=3000, tol=1e-6):
    curvature = np.array([1., kappa]); x = np.array([1., 1.]); velocity = np.zeros(2)
    for step in range(steps):
        velocity = beta * velocity - lr * curvature * x
        x = x + velocity
        size = np.linalg.norm(x)
        if size < tol: return step + 1
        if size > 1e6: return 'diverged'
    return 'not finished'
for kappa in [1., 10., 100.]:
    print(f'kappa {kappa:5.0f}: lr just below the limit {run(kappa, 1.9 / kappa)}, just above {run(kappa, 2.1 / kappa)}')
assert run(100., 2.1 / 100) == 'diverged'
plain, momentum = run(100., 1 / 100), run(100., 1 / 100, beta=.9)
print('kappa 100, lr 0.01: plain steps', plain, ' with momentum 0.9:', momentum)
assert momentum < plain


kappa     1: lr just below the limit 135, just above diverged
kappa    10: lr just below the limit 132, just above diverged
kappa   100: lr just below the limit 721, just above diverged
kappa 100, lr 0.01: plain steps 1375  with momentum 0.9: 264


## Part 3: Recover a filter by least squares
You know filters from Module 09 and least squares from Module 03. Here an unknown 3x3 kernel blurred some images, and you must recover it from input and output pairs. The inputs are smooth, because neighbouring pixels in real images are correlated (Module 06), and that correlation is what stretches this bowl. The Hessian is known exactly, so you can compute the largest stable learning rate rather than guess.


In [3]:
truth = np.array([[0.,.2,0.],[.2,.2,.2],[0.,.2,0.]])
noise = rng.normal(0, 1, (30, 18, 18))
images = np.lib.stride_tricks.sliding_window_view(noise, (3, 3), axis=(1, 2)).mean(axis=(3, 4)) * 3
patches = np.lib.stride_tricks.sliding_window_view(images, (3, 3), axis=(1, 2)).reshape(-1, 9)
targets = patches @ truth.ravel() + rng.normal(0, .01, len(patches))
exact = np.linalg.lstsq(patches, targets, rcond=None)[0]
print('samples:', patches.shape, ' recovered kernel:\n', exact.reshape(3, 3).round(3))
loss = lambda w: np.mean((patches @ w - targets) ** 2)
gradient = lambda w, A=patches, b=targets: 2 * A.T @ (A @ w - b) / len(A)
hessian = 2 * patches.T @ patches / len(patches)
curvatures = np.linalg.eigvalsh(hessian)
print('Hessian eigenvalues from', curvatures[0].round(3), 'to', curvatures[-1].round(3), '; all positive, so this objective is convex')
print('condition number, the stretch of the bowl:', (curvatures[-1] / curvatures[0]).round(1))
assert curvatures[0] > 0 and curvatures[-1] / curvatures[0] > 20
limit = 2 / curvatures[-1]
print('largest stable full-batch learning rate:', limit)
def descend(lr, beta=0., steps=300):
    w = np.zeros(9); v = np.zeros(9); curve = []
    for _ in range(steps):
        curve.append(loss(w)); v = beta * v - lr * gradient(w); w = w + v
    return w, curve
print('setting      loss at steps 0, 20, 100, 299      largest error in the kernel')
for name, lr, beta in [('too small', .05 * limit, 0.), ('near limit', .9 * limit, 0.), ('momentum', .25 * limit, .9), ('too large', 1.05 * limit, 0.)]:
    w, curve = descend(lr, beta)
    print(f'{name:10s} {curve[0]:9.4f} {curve[20]:9.4f} {curve[100]:9.5f} {curve[299]:11.4g}   {np.abs(w - exact).max():.3g}')
assert np.abs(descend(.25 * limit, .9)[0] - exact).max() < np.abs(descend(.9 * limit)[0] - exact).max()
assert descend(1.05 * limit)[1][-1] > descend(1.05 * limit)[1][0]


samples: (5880, 9)  recovered kernel:
 [[-0.     0.2    0.   ]
 [ 0.2    0.201  0.199]
 [ 0.     0.2    0.   ]]
Hessian eigenvalues from 0.084 to 7.277 ; all positive, so this objective is convex
condition number, the stretch of the bowl: 86.4
largest stable full-batch learning rate: 0.27484386970889274
setting      loss at steps 0, 20, 100, 299      largest error in the kernel
too small     0.5055    0.0190   0.00263   0.0006565   0.0622
near limit    0.5055    0.0006   0.00012   0.0001006   0.00017
momentum      0.5055    0.0454   0.00010   0.0001006   2.58e-08
too large     0.5055   22.0631 92573379.62628   2.759e+24   4.1e+11


## Part 4: Mini-batches and adaptive steps
A mini-batch gradient is a noisy estimate of the full gradient, so the loss jitters while each step stays cheap. Adam scales each coordinate by a running estimate of its gradient size. Compare them on the same data, from the same start, and read the curves rather than only the final number.


In [4]:
def minibatch(lr, batch=32, epochs=8):
    w = np.zeros(9); curve = []; generator = np.random.default_rng(1)
    for _ in range(epochs):
        order = generator.permutation(len(patches))
        for start in range(0, len(order), batch):
            chosen = order[start:start + batch]
            w -= lr * gradient(w, patches[chosen], targets[chosen])
        curve.append(loss(w))
    return w, curve
def adam(lr=.02, steps=300, b1=.9, b2=.999):
    w = np.zeros(9); m = np.zeros(9); s = np.zeros(9)
    for t in range(1, steps + 1):
        g = gradient(w); m = b1 * m + (1 - b1) * g; s = b2 * s + (1 - b2) * g * g
        w = w - lr * (m / (1 - b1 ** t)) / (np.sqrt(s / (1 - b2 ** t)) + 1e-8)
    return w
w, curve = minibatch(.4 * limit)
print('mini-batch loss after each epoch:', np.round(curve, 5), ' updates per epoch:', int(np.ceil(len(patches) / 32)))
print('exact least-squares loss:', loss(exact))
assert curve[-1] < 2 * loss(exact)
print('Adam loss after 300 steps:', loss(adam()))
assert loss(adam()) < 10 * loss(exact)


mini-batch loss after each epoch: [0.00014 0.00011 0.0001  0.0001  0.00011 0.0001  0.0001  0.00011]  updates per epoch: 184
exact least-squares loss: 0.00010060031719775125
Adam loss after 300 steps: 0.00010105104303728219


## Part 5: A stationary point is not automatically a minimum
A zero gradient says only that the surface is flat there. The Hessian tells you what kind of flat. On a nonconvex function, where you start decides where you finish, so no single run proves a global minimum.


In [5]:
f = lambda x: x ** 4 - 3 * x ** 2 + x
df = lambda x: 4 * x ** 3 - 6 * x + 1
d2f = lambda x: 12 * x ** 2 - 6
for start in [-2., 2.]:
    x = start
    for _ in range(200): x -= .01 * df(x)
    print(f'start {start:4.1f} ends at x={x:.4f}, f={f(x):.4f}, gradient {df(x):.2e}, curvature {d2f(x):.2f}')
left, right = -2., 2.
for _ in range(200): left -= .01 * df(left); right -= .01 * df(right)
assert abs(df(left)) < 1e-4 and abs(df(right)) < 1e-4 and f(left) < f(right) - 1


start -2.0 ends at x=-1.3008, f=-3.5139, gradient -1.88e-13, curvature 14.31
start  2.0 ends at x=1.1309, f=-1.0702, gradient 9.28e-09, curvature 9.35


## Your experiment
Pick one setting from Part 3 that fails. Say what the loss curve shows, explain the failure from the Hessian, and fix it by changing one thing. Then compare momentum values with everything else fixed. Keep your learning-rate table: in Module 07 you will use it to train a classifier, without guessing.
